# TKAN × Momentum — Combined Signals Review

| Section | Content |
|---------|---------|
| **1. Load Data** | CACT TR from DB; TKAN pred_cache + signal_cache; clip to OOS |
| **2. TKAN Signal** | Signal build + 1× MOO equity |
| **3. Momentum Signal** | Interactive θ slider, MOO execution |
| **4. Side by Side** | TKAN 1× vs Momentum 1× vs B&H (all MOO, no leverage) |
| **5. Combined** | AND / OR / regime-filter combinations |

In [9]:
import os, sys, pickle, warnings, importlib
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# ── Resolve paths ──────────────────────────────────────────────────────────
try:
    _HERE = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _HERE = os.path.abspath('')

_ROOT     = os.path.normpath(os.path.join(_HERE, *(['..'] * 7)))
_SFERA_DB = os.path.join(_ROOT, 'sfera-db')
_SIGNUM   = os.path.join(_ROOT, 'signum')

for _p in [_SFERA_DB, _SIGNUM]:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

import sfera_db
import signum.engine.dashboard, signum.engine.chart, signum
importlib.reload(signum.engine.chart)
importlib.reload(signum.engine.dashboard)
importlib.reload(signum)
from signum import Chart, Dashboard
from signum.engine.chart import Chart as _Chart

WEIGHTS_DIR     = os.path.join(_HERE, 'v4', 'weights')
ENTRY_THRESHOLD = 1.015        # TKAN: max(d1..d10) >= 1.5% → long
OOS_START       = '2015-01-02' # cycle_000 retrain_date from manifest.json

## 1. Load Data


In [10]:
# ── CACT Total Return (OHLC — open needed for MOO mode) ──────────────────
cactr = (sfera_db.query(
    "SELECT trade_date AS date, open_price AS open, close_price AS close "
    "FROM bbgidx.index_total_return WHERE ticker='CACT' ORDER BY trade_date")
    .assign(date=lambda d: pd.to_datetime(d['date']))
    .set_index('date'))

# ── TKAN v4 prediction cache ──────────────────────────────────────────────
with open(os.path.join(WEIGHTS_DIR, 'pred_cache.pkl'), 'rb') as f:
    pred_df = pickle.load(f)
pred_df.index = pd.DatetimeIndex(pred_df.index)
pred_cols = sorted([c for c in pred_df.columns if c.startswith('d') and c[1:].isdigit()],
                   key=lambda c: int(c[1:]))

print(f'CACT TR   : {cactr.index[0].date()} → {cactr.index[-1].date()}  ({len(cactr):,} rows)')
print(f'TKAN preds: {pred_df.index[0].date()} → {pred_df.index[-1].date()}  ({len(pred_df):,} rows)  cols={pred_cols}')
print(f'Open coverage: {cactr["open"].notna().mean():.1%}')


CACT TR   : 2000-01-03 → 2026-04-17  (6,725 rows)
TKAN preds: 2015-01-02 → 2026-03-24  (2,873 rows)  cols=['d1', 'd2', 'd3', 'd4', 'd5', 'd6', 'd7', 'd8', 'd9', 'd10']
Open coverage: 100.0%


## 2. TKAN Signal (1× MOO)

In [11]:
# ── Common dates & returns ─────────────────────────────────────────────────
common      = cactr.index.intersection(pred_df.index)
common      = common[common >= pd.Timestamp(OOS_START)]
common.name = 'date'
cact_close  = cactr.loc[common, 'close']
cact_open   = cactr.loc[common, 'open']
log_ret     = np.log(cact_close / cact_close.shift(1)).fillna(0)
cc_ret      = np.exp(log_ret) - 1
oc_ret      = (cact_close / cact_open - 1).fillna(0)

# ── TKAN signal (state-machine from walk-forward pipeline) ────────────────
# Entry: max(d1..d10) >= 1.015 on 2× lev series, hold until target hit, re-enter on exit bar
_sig_path = os.path.join(WEIGHTS_DIR, 'signal_cache.pkl')
with open(_sig_path, 'rb') as f:
    tkan_signal = pickle.load(f).reindex(common).fillna(0).astype(int)
print(f'TKAN signal: {common[0].date()}→{common[-1].date()} | in-market {tkan_signal.mean():.0%}')

TKAN signal: 2015-01-02→2026-03-24 | in-market 88%


In [12]:
# ── TKAN 1× MOO equity ─────────────────────────────────────────────────────
r_tkan_moo = _Chart.apply_execution(tkan_signal.astype(float), cc_ret,
                                     execution='NO', open_returns=oc_ret, carry_in=True)
lr_tkan = np.log(1 + r_tkan_moo).fillna(0)
eq_tkan = np.exp(lr_tkan.cumsum()).rename('value')
eq_bh   = np.exp(log_ret.cumsum()).rename('value')

ann = lr_tkan.mean()*252; vol = lr_tkan.std()*252**0.5
eq_ = np.exp(np.cumsum(np.asarray(lr_tkan)))
dd  = (eq_ / np.maximum.accumulate(eq_) - 1).min()
print(f'TKAN 1× MOO | {eq_[-1]:.2f}× | CAGR {ann*100:+.1f}% | Vol {vol*100:.1f}% | '
      f'Sharpe {ann/vol:.2f} | MaxDD {dd*100:.1f}% | in-market {tkan_signal.mean():.0%}')

_pos = tkan_signal.shift(1).fillna(0).astype(int).to_frame('position')
Dashboard(
    panes=[
        (Chart(theme='midnight', height=250)
         .line(eq_tkan, name='TKAN 1× MOO', color='#00e676', width=2)
         .line(eq_bh,   name='B&H CACT',    color='#5b9cf6', width=1)
         .shade(_pos, position_col='position', color='#00e676', opacity=0.10)),
        (Chart(theme='midnight', height=70)
         .line(tkan_signal.rename('value').astype(float), name='TKAN signal', color='#00e676', width=1)),
    ],
    titles=['TKAN 1× MOO equity vs B&H', 'TKAN signal (0/1)'],
    theme='midnight',
).show()

TKAN 1× MOO | 2.97× | CAGR +9.6% | Vol 18.1% | Sharpe 0.53 | MaxDD -38.6% | in-market 88%


## 3. Momentum Signal


### 3a. Momentum Slider — Interactive θ Sweep

Drag the slider to explore how threshold θ affects the **MOO** (next-open, intraday) equity curve.

**Why MOO underperforms MOC for CACT:** `oc_ret = close/open − 1` captures only the intraday
component. CACT is a bond total-return index — most return accrues overnight (coupon/carry gap
from close→open). MOO misses this entirely, so it is structurally weak versus MOC. This is correct and expected.

`carry_in=True`: position carried in if signal was already on before series start.


In [13]:
# ── Momentum parameters ───────────────────────────────────────────────────
MOM_WINDOW = 139
MOM_THETA  = 0.08
CARRY_IN   = True

mom_f    = np.log(cact_close / cact_close.shift(MOM_WINDOW))   # raw factor (NaN for first w bars)
gate_raw = (mom_f >= MOM_THETA).astype(float)

# ── Both execution modes (MOC = exec 1; MOO = exec 'NO') ──
r_moc = _Chart.apply_execution(gate_raw, cc_ret, execution=1, carry_in=CARRY_IN)
r_moo = _Chart.apply_execution(gate_raw, cc_ret, execution='NO', open_returns=oc_ret, carry_in=CARRY_IN)

# MOO log-returns used in the diagnostic section
mom_ret      = np.log(1 + r_moo).fillna(0)
nav_moc_full = (1 + r_moc).cumprod()
nav_moo_full = (1 + r_moo).cumprod()
nav_bh_full  = (1 + cc_ret).cumprod()

# ── Momentum MOO threshold slider (equity base = 1.0) ─────────────────────
_theta_min = round(float(mom_f.dropna().min()) - 0.005, 3)
_theta_max = round(float(mom_f.dropna().max()) + 0.005, 3)

(Dashboard(
    panes=[
        Chart(theme='midnight', height=260),
        (Chart(theme='midnight', height=150)
         .baseline(mom_f.rename('value').dropna(), base_value=MOM_THETA,
                   name=f'Momentum({MOM_WINDOW})',
                   topLineColor='#ffd600',
                   topFillColor1='rgba(255,214,0,0.12)', topFillColor2='transparent',
                   bottomLineColor='rgba(140,140,140,0.6)',
                   bottomFillColor1='rgba(140,140,140,0.05)', bottomFillColor2='transparent')),
    ],
    titles=[
        f'Equity (base=1.0)  |  Momentum MOO  w={MOM_WINDOW}  carry_in={CARRY_IN}  (gold) vs B&H close→close (blue)',
        f'Momentum({MOM_WINDOW}) vs threshold θ',
    ],
    theme='midnight',
    execution='NO',
).threshold_control(
    df=mom_f.to_frame('value'),
    signal_mode='>=',
    threshold=MOM_THETA,
    min_val=_theta_min, max_val=_theta_max, step=0.005,
    price_pane=1, equity_pane=0,
    prices=cactr.loc[common],
    strategy_color='#ffd600',
    bh_color='#5b9cf6',
    equity_base=1.0,
    carry_in=CARRY_IN,
).show())

## 4. TKAN × Momentum — Side by Side

TKAN 1× vs Momentum 1× vs B&H, all MOO execution. No leverage.

In [14]:
# ── Stats helper ───────────────────────────────────────────────────────────
def _qs(log_r, name, sig=None):
    lr = np.asarray(log_r); ann = lr.mean()*252; vol = lr.std()*252**0.5
    eq_ = np.exp(np.cumsum(lr)); dd = (eq_/np.maximum.accumulate(eq_)-1).min()
    pct = f'{float(sig.mean()):.0%}' if sig is not None else '100%'
    return dict(Strategy=name, Total=f'{eq_[-1]:.2f}×', CAGR=f'{ann*100:+.1f}%',
                Vol=f'{vol*100:.1f}%', Sharpe=f'{ann/vol:.2f}' if vol else 'n/a',
                MaxDD=f'{dd*100:.1f}%', InMkt=pct)

_mom_log = mom_ret.reindex(common).fillna(0)
_mom_eq  = np.exp(_mom_log.cumsum()).rename('value')

display(pd.DataFrame([
    _qs(lr_tkan, 'TKAN 1× MOO', tkan_signal),
    _qs(_mom_log, f'Mom 1× MOO (θ={MOM_THETA}, w={MOM_WINDOW})', gate_raw),
    _qs(log_ret, 'B&H CACT'),
]).set_index('Strategy'))

# ── Dashboard ─────────────────────────────────────────────────────────────
_tkan_pos = tkan_signal.shift(1).fillna(0).astype(int).to_frame('position')
Dashboard(
    panes=[
        (Chart(theme='midnight', height=280, watermark='1× MOO equity (base=1.0)')
         .line(eq_tkan, name='TKAN 1× MOO',   color='#00e676', width=2)
         .line(_mom_eq, name=f'Mom 1× MOO',    color='#ffd600', width=1)
         .line(eq_bh,   name='B&H CACT',       color='#5b9cf6', width=1)
         .shade(_tkan_pos, position_col='position', color='#00e676', opacity=0.08)),
        (Chart(theme='midnight', height=70)
         .line(tkan_signal.rename('value').astype(float), name='TKAN',     color='#00e676', width=1)
         .line(gate_raw.rename('value'),                  name='Momentum', color='#ffd600', width=1)),
    ],
    titles=[
        f'TKAN vs Momentum vs B&H — all 1× MOO',
        'Signals: TKAN (green) / Momentum (gold)',
    ],
    theme='midnight',
).show()

,Total,CAGR,Vol,Sharpe,MaxDD,InMkt
Strategy,,,,,,
TKAN 1× MOO,2.97×,+9.6%,18.1%,0.53,-38.6%,88%
"Mom 1× MOO (θ=0.08, w=139)",1.34×,+2.5%,8.2%,0.31,-18.9%,36%
B&H CACT,2.55×,+8.2%,18.5%,0.45,-38.6%,100%


## 5. Combined Signal — AND (TKAN ∩ Momentum)

MOO execution (`carry_in=True`). Long only when TKAN=1 **and** momentum ≥ θ.

In [15]:
# ── AND signal: TKAN ∩ Momentum ────────────────────────────────────────────
mom_gate = (mom_f.reindex(common) >= MOM_THETA).astype(int).fillna(0)
sig_and  = (tkan_signal & mom_gate).astype(float)

r_and  = _Chart.apply_execution(sig_and, cc_ret, execution='NO', open_returns=oc_ret, carry_in=CARRY_IN)
r_tkan = _Chart.apply_execution(tkan_signal.astype(float), cc_ret, execution='NO', open_returns=oc_ret, carry_in=CARRY_IN)
r_mom  = _Chart.apply_execution(mom_gate.astype(float), cc_ret, execution='NO', open_returns=oc_ret, carry_in=CARRY_IN)

eq_and  = (1 + r_and).cumprod().rename('value')
eq_tkan = (1 + r_tkan).cumprod().rename('value')
eq_mom  = (1 + r_mom).cumprod().rename('value')
eq_bh   = (1 + cc_ret).cumprod().rename('value')

lr_and  = np.log(1 + r_and).fillna(0)
lr_tkan = np.log(1 + r_tkan).fillna(0)
lr_mom  = np.log(1 + r_mom).fillna(0)

display(pd.DataFrame([
    _qs(lr_and,  'AND (TKAN ∩ Mom)', sig_and),
    _qs(lr_tkan, 'TKAN only MOO',    tkan_signal),
    _qs(lr_mom,  f'Mom only MOO (θ={MOM_THETA})', mom_gate),
    _qs(log_ret, 'B&H CACT'),
]).set_index('Strategy'))

print(f'TKAN=1: {int(tkan_signal.sum()):,}d  Mom=1: {int(mom_gate.sum()):,}d  '
      f'Both=1: {int((tkan_signal & mom_gate).sum()):,}d ({(tkan_signal & mom_gate).mean():.0%})')

,Total,CAGR,Vol,Sharpe,MaxDD,InMkt
Strategy,,,,,,
AND (TKAN ∩ Mom),1.48×,+3.4%,7.7%,0.44,-13.8%,29%
TKAN only MOO,2.97×,+9.6%,18.1%,0.53,-38.6%,88%
Mom only MOO (θ=0.08),1.34×,+2.5%,8.2%,0.31,-18.9%,36%
B&H CACT,2.55×,+8.2%,18.5%,0.45,-38.6%,100%


TKAN=1: 2,533d  Mom=1: 1,042d  Both=1: 842d (29%)


In [16]:
# ── AND signal dashboard ────────────────────────────────────────────────────
_and_pos = sig_and.shift(1).fillna(0).astype(int).to_frame('position')
Dashboard(
    panes=[
        (Chart(theme='midnight', height=280, watermark='MOO equity (base=1.0)')
         .line(eq_and,  name='AND (TKAN ∩ Mom)', color='#00e676', width=2)
         .line(eq_tkan, name='TKAN only',        color='#bb86fc', width=1)
         .line(eq_mom,  name='Mom only',         color='#ffd600', width=1)
         .line(eq_bh,   name='B&H CACT',        color='#5b9cf6', width=1)
         .shade(_and_pos, position_col='position', color='#00e676', opacity=0.08)),
        (Chart(theme='midnight', height=70)
         .line(tkan_signal.rename('value').astype(float), name='TKAN',     color='#bb86fc', width=1)
         .line(mom_gate.rename('value').astype(float),    name='Momentum', color='#ffd600', width=1)
         .line(sig_and.rename('value'),                   name='AND',      color='#00e676', width=1)),
    ],
    titles=['AND (TKAN ∩ Mom) vs individual — MOO execution', 'Signals'],
    theme='midnight',
).show()